In [1]:
pip install biopython pyfaidx pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
def get_sequence(chromosome, position, flank=12):
    """
    Extract a sequence of length (2*flank + 1) around the SNP position.

    Parameters:
    - chromosome: Chromosome name (e.g., 'chr1')
    - position: SNP position (1-based coordinate)
    - flank: Number of bases upstream and downstream of the SNP

    Returns:
    - sequence: DNA sequence as a string
    """
    # Adjust for 0-based indexing in pyfaidx
    start = position - flank - 1  # Subtract 1 because positions are 1-based
    end = position + flank        # End position is exclusive in slicing

    # Handle cases where start is negative
    if start < 0:
        start = 0

    # Extract the sequence
    try:
        seq = genome[chromosome][start:end]
        return str(seq).upper()
    except KeyError:
        print(f"Chromosome {chromosome} not found in the reference genome.")
        return None
    except Exception as e:
        print(f"Error retrieving sequence for {chromosome}:{position}: {e}")
        return None


# Define the padding function
def pad_sequence(seq, target_length=500):
    """
    Pads the input sequence with 'N's on both sides to reach the target length.
    The original sequence is centered. If the sequence is longer than target_length,
    it is truncated from the center.
    
    Parameters:
        seq (str): The original DNA sequence.
        target_length (int): The desired total length after padding.
    
    Returns:
        str: The padded (or truncated) sequence.
    """
    seq_length = len(seq)
    
    if seq_length >= target_length:
        # If the sequence is longer than or equal to target_length, truncate it
        # to the target_length by keeping the center part
        start = (seq_length - target_length) // 2
        end = start + target_length
        return seq[start:end]
    
    # Calculate total padding needed
    total_pad = target_length - seq_length
    
    # Calculate padding on each side
    pad_left = total_pad // 2
    pad_right = total_pad - pad_left
    
    # Create padded sequence
    padded_seq = 'N' * pad_left + seq + 'N' * pad_right
    return padded_seq

In [12]:
# Load SNP data
from pyfaidx import Fasta
from Bio.Seq import Seq
from Bio import SeqIO
import os
import pandas as pd
snp_data = pd.read_csv('indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv')
snp_data = snp_data[['RSID','chr','pos_hg38','Major','Minor','a1','a2']]#.drop_duplicates('RSID')
snp_data.columns = ['SNP_ID','chromosome','position','Major','Minor','a1','a2']
# Load the genome
genome = Fasta('/media/zihengc/T7/genome/hg38.fa' , as_raw=True)  # as_raw=True returns sequences as strings
# Create a new column for sequences
snp_data['Sequence'] = snp_data.apply(
    lambda row: get_sequence(row['chromosome'], row['position'], flank=12),
    axis=1
)
# Preview the updated DataFrame
missing_sequences = snp_data[snp_data['Sequence'].isnull()]
if not missing_sequences.empty:
    print("Sequences could not be retrieved for the following SNPs:")
    print(missing_sequences)
snp_data['hg38_center_base']=snp_data['Sequence'].str.slice(12,13)
snp_data['Major_len'] = snp_data['Major'].str.len()
snp_data['Minor_len'] = snp_data['Minor'].str.len()
# Create boolean mask
mask = (snp_data['Major'].str.len() == 1) & (snp_data['Minor'].str.len() == 1)
snp_data_single = snp_data[mask].copy()
# Create a boolean mask where 'Seq_13' matches 'Major' or 'Minor'
mask = snp_data_single.apply(
    lambda row: (row['hg38_center_base'] == row['Major']) or (row['hg38_center_base'] == row['Minor']),
    axis=1
)
# Filter the DataFrame
snp_data_single = snp_data_single[mask].copy()
snp_data_single['Sequence_Major'] = snp_data_single['Sequence'].str.slice(0,12)+snp_data_single['Major']+snp_data_single['Sequence'].str.slice(13,)
snp_data_single['Sequence_Minor'] = snp_data_single['Sequence'].str.slice(0,12)+snp_data_single['Minor']+snp_data_single['Sequence'].str.slice(13,)

snp_data_multi = snp_data.loc[~snp_data.index.isin(snp_data_single.index)]
snp_data_multi

,SNP_ID,chromosome,position,Major,Minor,a1,a2,Sequence,hg38_center_base,Major_len,Minor_len
143,rs1198397268,chr19,1999196,CAG,C,CAG,C,TTGACTGCACCCCAGAGTCAGGTCA,C,3,1
189,rs1394008105,chr2,104659015,G,GT,G,GT,TCTTCCCCCAGCGTATTTCCTATGG,G,1,2
190,rs1394008105,chr2,104659015,G,GT,G,GT,TCTTCCCCCAGCGTATTTCCTATGG,G,1,2
374,rs34158903,chr15,58727205,TC,T,TC,T,TTTTTTTTTTTTTCCCAAAAACAGC,T,2,1
375,rs34158903,chr15,58727205,TC,T,TC,T,TTTTTTTTTTTTTCCCAAAAACAGC,T,2,1
491,rs55706526,chr4,11039925,A,AAT,AAT,A,TGGGAAACAACCAATATCAATCAGC,A,1,3
572,rs67141839,chr6,32604799,G,GGACAT,GGACAT,G,TAAATTAGGACATGACATAGGTGGT,T,1,6
602,rs71705525,chr6,32608617,A,AACG,AACG,A,TATAAGAAGAACGTGCAGGGACAAT,G,1,4
845,rs9281938,chr6,32608505,TA,T,T,TA,TTGCTGTAATGCTGGATCTCATCAC,T,2,1
852,rs965034941,chr19,1999195,CCA,C,CCA,C,CTTGACTGCACCCCAGAGTCAGGTC,C,3,1


In [ ]:
snp_data['hg38_center_base']=snp_data['Sequence'].str.slice(12,13)
snp_data['Major_len'] = snp_data['Major'].str.len()
snp_data['Minor_len'] = snp_data['Minor'].str.len()
# Create boolean mask
mask = (snp_data['Major'].str.len() == 1) & (snp_data['Minor'].str.len() == 1)
snp_data_single = snp_data[mask].copy()
# Create a boolean mask where 'Seq_13' matches 'Major' or 'Minor'
mask = snp_data_single.apply(
    lambda row: (row['hg38_center_base'] == row['Major']) or (row['hg38_center_base'] == row['Minor']),
    axis=1)
# Filter the DataFrame
snp_data_single = snp_data_single[mask].copy()
snp_data_single['Sequence_Major'] = snp_data_single['Sequence'].str.slice(0,12)+snp_data_single['Major']+snp_data_single['Sequence'].str.slice(13,)
snp_data_single['Sequence_Minor'] = snp_data_single['Sequence'].str.slice(0,12)+snp_data_single['Minor']+snp_data_single['Sequence'].str.slice(13,)
snp_data_multi = snp_data.loc[~snp_data.index.isin(snp_data_single.index)]
#snp_data_multi.to_csv('MPRA_SNPCenter_Sequences_INDEL_NeedManualAnnotation.csv')

######################mannually modified this snp_data_multi.csv########################################
snp_data_multi_modified = pd.read_csv('allele_differences_motif_analysis/MPRA_SNPCenter_Sequences_INDEL_withManualAnnotation.csv',index_col=0)
snp_data_modified = pd.concat([snp_data_single,snp_data_multi_modified]).sort_index()
# Apply the padding function to 'Sequence_Major' and 'Sequence_Minor'
snp_data_modified['Sequence_Major_padded'] = snp_data_modified['Sequence_Major'].apply(pad_sequence)
snp_data_modified['Sequence_Minor_padded'] = snp_data_modified['Sequence_Minor'].apply(pad_sequence)
snp_data_modified.to_csv("indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv")

In [28]:
snp_data_modified=pd.read_csv("indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv",index_col=0)
snp_data_modified

,SNP_ID,chromosome,position,Major,Minor,a1,a2,Sequence,hg38_center_base,Major_len,Minor_len,Sequence_Major,Sequence_Minor,Padded_Sequence,Is_Valid_Length,Sequence_Major_padded,Sequence_Minor_padded
0,cg03073402,chr19,42423524,C,G,C,G,GCTCCGTTCCCCCGGCCAACTCCAT,C,1,1,GCTCCGTTCCCCCGGCCAACTCCAT,GCTCCGTTCCCCGGGCCAACTCCAT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
1,cg03169557,chr16,89532542,C,G,C,G,CATCGATGAGATCGACGCGGTGGGC,C,1,1,CATCGATGAGATCGACGCGGTGGGC,CATCGATGAGATGGACGCGGTGGGC,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
2,cg05030077,chr16,2205198,C,G,C,G,CAGGGAGGGCACCGGAACAGCGCGA,C,1,1,CAGGGAGGGCACCGGAACAGCGCGA,CAGGGAGGGCACGGGAACAGCGCGA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
3,cg05066959,chr8,41661790,C,G,C,G,CAGGGCCGCGGGCGCCTCAGTACCT,C,1,1,CAGGGCCGCGGGCGCCTCAGTACCT,CAGGGCCGCGGGGGCCTCAGTACCT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
4,cg05228284,chr19,2720849,C,G,C,G,AGAGGCACAGTGCGGGGCCAGCCCA,C,1,1,AGAGGCACAGTGCGGGGCCAGCCCA,AGAGGCACAGTGGGGGGCCAGCCCA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,rs9478143,chr6,150862035,A,G,A,G,GTGGCTGGAGGCAAGATATGCTGGC,A,1,1,GTGGCTGGAGGCAAGATATGCTGGC,GTGGCTGGAGGCGAGATATGCTGGC,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
851,rs953471,chr9,124221903,G,A,G,A,CAAAGGGTGGGTGGAGCTGTCGAGA,G,1,1,CAAAGGGTGGGTGGAGCTGTCGAGA,CAAAGGGTGGGTAGAGCTGTCGAGA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
852,rs965034941,chr19,1999195,CCA,C,CCA,C,CTTGACTGCACCCCAGAGTCAGGTC,C,3,1,CTTGACTGCACCCCAGAGTCAGGTC,CTTGACTGCACCCGAGTCAGGTC,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,True,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
853,rs983392,chr11,60156035,A,G,A,G,CTAAAATTCCACATGCAATTTTATT,A,1,1,CTAAAATTCCACATGCAATTTTATT,CTAAAATTCCACGTGCAATTTTATT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...


# Major vs Minor in significant alleles

In [22]:
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Define a function to create SeqRecord objects for both alleles
def create_seq_records(df, condition=None):
    """
    Creates SeqRecord objects for both Major and Minor alleles based on a condition.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing SNP data.
        condition (callable, optional): A function to filter the DataFrame rows.
        
    Returns:
        list: A list of SeqRecord objects.
    """
    seq_records = []
    seq_major = []
    seq_minor = []

    # Apply the condition if provided
    if condition:
        data_to_save = df[condition]#.drop_duplicates('RSID')
    else:
        data_to_save = df#.drop_duplicates('RSID')
    
    for index, row in data_to_save.iterrows():
        rsid = row['RSID']
        chr_ = row['chr']
        pos = row['pos_hg38']
        major_base = row['Major']
        minor_base = row['Minor']
        
        # Padded sequences
        seq_major_padded = row.get('Sequence_Major_padded', row.get('Sequence_Major'))
        seq_minor_padded = row.get('Sequence_Minor_padded', row.get('Sequence_Minor'))
        
        # Create SeqRecord for Major allele
        if pd.notnull(seq_major_padded):
            major_id = f"{rsid}_Major_{major_base}"
            major_description = f"{chr_}:{pos}"
            record_major = SeqRecord(
                Seq(seq_major_padded),
                id=major_id,
                description=major_description
            )
            seq_records.append(record_major)
            seq_major.append(record_major)
        # Create SeqRecord for Minor allele
        if pd.notnull(seq_minor_padded):
            minor_id = f"{rsid}_Minor_{minor_base}"
            minor_description = f"{chr_}:{pos}"
            record_minor = SeqRecord(
                Seq(seq_minor_padded),
                id=minor_id,
                description=minor_description
            )
            seq_records.append(record_minor)
            seq_minor.append(record_minor)
    return seq_records,seq_major,seq_minor


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240813_comparative_THP1Macrophage_alleleOnly.csv', index_col=0)
snp_data_multi_modified = pd.read_csv('indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv',index_col=0)
df_allele = df_allele.join(snp_data_multi_modified.drop_duplicates('SNP_ID').set_index('SNP_ID')[['Sequence_Major_padded', 'Sequence_Minor_padded']],on='RSID',how='left')
seq_records_all = create_seq_records(df_allele)

############################################################################################
seq_records_sig,seq_major,seq_minor = create_seq_records(df_allele, condition=lambda df: df['fdr'] <= 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/THP1_macrophage_sig_major_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_major, fasta_file, 'fasta')

# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/THP1_macrophage_sig_minor_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_minor, fasta_file, 'fasta')

df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly.csv',index_col=0)
snp_data_multi_modified = pd.read_csv('indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv',index_col=0)
df_allele = df_allele.join(snp_data_multi_modified.drop_duplicates('SNP_ID').set_index('SNP_ID')[['Sequence_Major_padded', 'Sequence_Minor_padded']],on='RSID',how='left')
seq_records_all = create_seq_records(df_allele)

############################################################################################
seq_records_sig,seq_major,seq_minor = create_seq_records(df_allele, condition=lambda df: df['fdr'] <= 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/mouse_brain_sig_major_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_major, fasta_file, 'fasta')

# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/mouse_brain_sig_minor_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_minor, fasta_file, 'fasta')

# Sig vs NonSig

In [46]:
import pandas as pd
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO

# Define a function to create SeqRecord objects for both alleles
def create_seq_records(df, condition=None):
    """
    Creates SeqRecord objects for both Major and Minor alleles based on a condition.
    
    Parameters:
        df (pd.DataFrame): The DataFrame containing SNP data.
        condition (callable, optional): A function to filter the DataFrame rows.
        
    Returns:
        list: A list of SeqRecord objects.
    """
    seq_records = []
    seq_major = []
    seq_minor = []

    # Apply the condition if provided
    if condition:
        data_to_save = df[condition]#.drop_duplicates('RSID')
    else:
        data_to_save = df#.drop_duplicates('RSID')
    
    for index, row in data_to_save.iterrows():
        rsid = row['RSID']
        chr_ = row['chr']
        pos = row['pos_hg38']
        major_base = row['Major']
        minor_base = row['Minor']
        
        # Padded sequences
        seq_major_padded = row.get('Sequence_Major_padded', row.get('Sequence_Major'))
        seq_minor_padded = row.get('Sequence_Minor_padded', row.get('Sequence_Minor'))
        
        # Create SeqRecord for Major allele
        if pd.notnull(seq_major_padded):
            major_id = f"{rsid}_Major_{major_base}"
            major_description = f"{chr_}:{pos}"
            record_major = SeqRecord(
                Seq(seq_major_padded),
                id=major_id,
                description=major_description
            )
            seq_records.append(record_major)
            seq_major.append(record_major)
        # Create SeqRecord for Minor allele
        if pd.notnull(seq_minor_padded):
            minor_id = f"{rsid}_Minor_{minor_base}"
            minor_description = f"{chr_}:{pos}"
            record_minor = SeqRecord(
                Seq(seq_minor_padded),
                id=minor_id,
                description=minor_description
            )
            seq_records.append(record_minor)
            seq_minor.append(record_minor)
    return seq_records#,seq_major,seq_minor


df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240813_comparative_THP1Macrophage_alleleOnly.csv', index_col=0)
snp_data_multi_modified = pd.read_csv('indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv',index_col=0)
df_allele = df_allele.join(snp_data_multi_modified.drop_duplicates('SNP_ID').set_index('SNP_ID')[['Sequence_Major_padded', 'Sequence_Minor_padded']],on='RSID',how='left')
seq_records_all = create_seq_records(df_allele)
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/MPRA_SNPCenter_25bp500bp_1710Sequences.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_all, fasta_file, 'fasta')

############################################################################################
seq_records_sig = create_seq_records(df_allele, condition=lambda df: df['fdr'] <= 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/THP1_macrophage_sig_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_sig, fasta_file, 'fasta')

seq_records_sig = create_seq_records(df_allele, condition=lambda df: df['fdr'] > 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/THP1_macrophage_nonsig_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_sig, fasta_file, 'fasta')

df_allele = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/allele_differences_withoutcontrol/20240817_allele_only_for_plotting/annotated_20240616_comparative_BrainR1R2merged20240404_alleleOnly.csv',index_col=0)
snp_data_multi_modified = pd.read_csv('indexing/MPRA_SNPCenter_Sequences_withManualAnnotation.csv',index_col=0)
df_allele = df_allele.join(snp_data_multi_modified.drop_duplicates('SNP_ID').set_index('SNP_ID')[['Sequence_Major_padded', 'Sequence_Minor_padded']],on='RSID',how='left')
seq_records_all = create_seq_records(df_allele)
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/MPRA_SNPCenter_25bp500bp_1710Sequences.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_all, fasta_file, 'fasta')

############################################################################################
seq_records_sig = create_seq_records(df_allele, condition=lambda df: df['fdr'] <= 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/mouse_brain_sig_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_sig, fasta_file, 'fasta')

seq_records_sig  = create_seq_records(df_allele, condition=lambda df: df['fdr'] > 0.05)
# Write to a FASTA file
with open('/media/zihengc/T7/mpra3_lib_analysis/mpra3_meme_scanning/enrichment/mouse_brain_nonsig_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records_sig, fasta_file, 'fasta')

In [24]:
df_cluster = pd.read_csv("allele_clustering/normalized_snp_mpra_cluster_20241102_5cluster_fixed_noGutmerged.csv",index_col=0)[['Cluster']]
df_index = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv',index_col=0)
snp_data_multi_modified['index'] = df_index.index
snp_data_multi_modified = snp_data_multi_modified.set_index('index')
snp_mac_reduce = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 3].index]
snp_mac_increase = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 2].index]


from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_yellowCluster_immuneLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_yellowredCluster_immuneHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_blueCluster_immuneHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_blueCluster_immuneLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

df_cluster = pd.read_csv("allele_clustering/normalized_snp_mpra_cluster_20241102_5cluster_fixed_noGutmerged.csv",index_col=0)[['Cluster']]
df_index = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv',index_col=0)
snp_data_multi_modified['index'] = df_index.index
snp_data_multi_modified = snp_data_multi_modified.set_index('index')
snp_mac_reduce = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 0].index]
snp_mac_increase = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 1].index]

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_redCluster_brainLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_redCluster_brainHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_greenCluster_brainHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_mpra_greenCluster_brainLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [25]:
df_cluster = pd.read_csv("allele_clustering/normalized_snp_ml_cluster_20241102_5cluster_fixed.csv",index_col=0)[['Cluster']]
df_index = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv',index_col=0)
snp_data_multi_modified['index'] = df_index.index
snp_data_multi_modified = snp_data_multi_modified.set_index('index')
snp_data_multi_modified.to_csv("allele_clustering/normalized_snp_ml_cluster_20241102_5cluster_fixed_withindex.csv")
snp_mac_reduce = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 1].index]
snp_mac_increase = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 2].index]

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_yellowCluster_immuneLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_yellowredCluster_immuneHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_blueCluster_immuneHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_blueCluster_immuneLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

df_cluster = pd.read_csv("allele_clustering/normalized_snp_ml_cluster_20241102_5cluster_fixed.csv",index_col=0)[['Cluster']]
df_index = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv',index_col=0)
snp_data_multi_modified['index'] = df_index.index
snp_data_multi_modified = snp_data_multi_modified.set_index('index')
snp_data_multi_modified.to_csv("allele_clustering/normalized_snp_ml_cluster_20241102_5cluster_fixed_withindex.csv")
snp_mac_reduce = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 3].index]
snp_mac_increase = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 4].index]

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_redCluster_brainLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_redCluster_brainHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Major_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Major_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_greenCluster_brainHigh_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row['Sequence_Minor_padded']):
        record = SeqRecord(
            Seq(row['Sequence_Minor_padded']),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

# Write to a FASTA file
with open('mpra3_meme_scanning/snp_atac_ml_greenCluster_brainLow_allele_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [26]:
from Bio import SeqIO
import pandas as pd

# Define the path to the fasta file
fasta_path = '/media/zihengc/T7/THP1_machinelearning/mpra3_prediction/sequence_for_prediction/20240618_MPRA_500bp_SNPonly_500bpbarcodes.fasta'

# Read the fasta file and convert it to a list of dictionaries
fasta_data = [{'ID': record.id, 'Sequence': str(record.seq)} for record in SeqIO.parse(fasta_path, 'fasta')]

# Convert the list of dictionaries to a DataFrame
fasta_df = pd.DataFrame(fasta_data)

# Define the output CSV path
#csv_output_path = '/mnt/data/20240618_MPRA_500bp_SNPonly_500bpbarcodes.csv'

# Save the DataFrame to a CSV file
#fasta_df.to_csv(csv_output_path, index=False)
fasta_df = fasta_df.set_index('ID')
# Display the first few rows of the DataFrame
fasta_df.head()


,Sequence
ID,
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1020750,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.843358,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1182770,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.931684,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1395605,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...


In [27]:
df_index=pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/MPRA3_Contributor_20231108_unique_GeneName_BarcodeEnhancerPair.csv',index_col=0)
df_index

,enhancer_id,barcode_id,barcode_seq,barcode_seq_reverse_complement,Type,enhancer_seq,full_seq,RSID,Contributor,nearest_gene,Barcode_RC_Enhancer10_R1,Barcode_RC_Enhancer10_R2,Barcode_RC_Enhancer10_R2_R1
ID,,,,,,,,,,,,,
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1020750,alt:rs139359594:SNPCENTER:chr5:151082227:A:G:1...,free16-1.1020750,GCAACCAACGCCTCCT,AGGAGGCGTTGGTTGC,AD,CCTCATTTTACAGAAGAGAGACCCATCAATCAACTGTGACTTCGCA...,GAGTCTGAACCTGTGTGCTACCTCATTTTACAGAAGAGAGACCCAT...,rs139359594,Hide,TNIP1,AGGAGGCGTTGGTTGC_CCTCCCTTCT,AGGAGGCGTTGGTTGC_CCTCATTTTA,AGGAGGCGTTGGTTGC_CCTCATTTTA_CCTCCCTTCT
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.843358,alt:rs139359594:SNPCENTER:chr5:151082227:A:G:1...,free16-1.843358,CTCCAGGTGTCAGCAT,ATGCTGACACCTGGAG,AD,CCTCATTTTACAGAAGAGAGACCCATCAATCAACTGTGACTTCGCA...,GAGTCTGAACCTGTGTGCTACCTCATTTTACAGAAGAGAGACCCAT...,rs139359594,Hide,TNIP1,ATGCTGACACCTGGAG_CCTCCCTTCT,ATGCTGACACCTGGAG_CCTCATTTTA,ATGCTGACACCTGGAG_CCTCATTTTA_CCTCCCTTCT
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1182770,alt:rs139359594:SNPCENTER:chr5:151082227:A:G:1...,free16-1.1182770,GGTTGGTAATGCTGGA,TCCAGCATTACCAACC,AD,CCTCATTTTACAGAAGAGAGACCCATCAATCAACTGTGACTTCGCA...,GAGTCTGAACCTGTGTGCTACCTCATTTTACAGAAGAGAGACCCAT...,rs139359594,Hide,TNIP1,TCCAGCATTACCAACC_CCTCCCTTCT,TCCAGCATTACCAACC_CCTCATTTTA,TCCAGCATTACCAACC_CCTCATTTTA_CCTCCCTTCT
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.931684,alt:rs139359594:SNPCENTER:chr5:151082227:A:G:1...,free16-1.931684,GAATCATAAGCCGAAT,ATTCGGCTTATGATTC,AD,CCTCATTTTACAGAAGAGAGACCCATCAATCAACTGTGACTTCGCA...,GAGTCTGAACCTGTGTGCTACCTCATTTTACAGAAGAGAGACCCAT...,rs139359594,Hide,TNIP1,ATTCGGCTTATGATTC_CCTCCCTTCT,ATTCGGCTTATGATTC_CCTCATTTTA,ATTCGGCTTATGATTC_CCTCATTTTA_CCTCCCTTCT
alt:rs139359594:SNPCENTER:chr5:151082227:A:G:151081882:151082382:151082132:free16-1.1395605,alt:rs139359594:SNPCENTER:chr5:151082227:A:G:1...,free16-1.1395605,TCAGACGTTGTAATGT,ACATTACAACGTCTGA,AD,CCTCATTTTACAGAAGAGAGACCCATCAATCAACTGTGACTTCGCA...,GAGTCTGAACCTGTGTGCTACCTCATTTTACAGAAGAGAGACCCAT...,rs139359594,Hide,TNIP1,ACATTACAACGTCTGA_CCTCCCTTCT,ACATTACAACGTCTGA_CCTCATTTTA,ACATTACAACGTCTGA_CCTCATTTTA_CCTCCCTTCT
...,...,...,...,...,...,...,...,...,...,...,...,...,...
ARP.Control.Nguyen_neg_T03000009843#free16-1.861448,ARP.Control.Nguyen_neg_T03000009843,free16-1.861448,CTCTTACCGCTAGTCC,GGACTAGCGGTAAGAG,Control,GGTACTCCCCTTTCTTACTTTCGGAAAGTCTATTGTTAAGCATTGT...,GAGTCTGAACCTGTGTGCTAGGTACTCCCCTTTCTTACTTTCGGAA...,ARP.Control.Nguyen_neg_T03000009843,Pfenning_control,NaN,GGACTAGCGGTAAGAG_GGTAAGTTTG,GGACTAGCGGTAAGAG_GGTACTCCCC,GGACTAGCGGTAAGAG_GGTACTCCCC_GGTAAGTTTG
ARP.Control.Nguyen_neg_T03000009843#free16-1.57071,ARP.Control.Nguyen_neg_T03000009843,free16-1.57071,AAGCAGAGTTAGCGCC,GGCGCTAACTCTGCTT,Control,GGTACTCCCCTTTCTTACTTTCGGAAAGTCTATTGTTAAGCATTGT...,GAGTCTGAACCTGTGTGCTAGGTACTCCCCTTTCTTACTTTCGGAA...,ARP.Control.Nguyen_neg_T03000009843,Pfenning_control,NaN,GGCGCTAACTCTGCTT_GGTAAGTTTG,GGCGCTAACTCTGCTT_GGTACTCCCC,GGCGCTAACTCTGCTT_GGTACTCCCC_GGTAAGTTTG
ARP.Control.Nguyen_neg_T03000009843#free16-1.911401,ARP.Control.Nguyen_neg_T03000009843,free16-1.911401,GAACACGGTAGAGCGG,CCGCTCTACCGTGTTC,Control,GGTACTCCCCTTTCTTACTTTCGGAAAGTCTATTGTTAAGCATTGT...,GAGTCTGAACCTGTGTGCTAGGTACTCCCCTTTCTTACTTTCGGAA...,ARP.Control.Nguyen_neg_T03000009843,Pfenning_control,NaN,CCGCTCTACCGTGTTC_GGTAAGTTTG,CCGCTCTACCGTGTTC_GGTACTCCCC,CCGCTCTACCGTGTTC_GGTACTCCCC_GGTAAGTTTG


In [28]:
fasta_df = pd.merge(fasta_df,df_index,left_index=True,right_index=True).drop_duplicates(subset=['enhancer_id'], keep='first')

In [29]:
fasta_df = fasta_df.set_index('enhancer_id')

In [30]:
df_alref = pd.read_csv("indexing/ALT_REF_LookUpTable_filtered_amended_alleleOnly_20240605.csv")

In [31]:
df_alref

,0,1
0,alt:rs9912783:PEAKCENTER:chr17:61522705:C:T:61...,ref:rs9912783:PEAKCENTER:chr17:61522705:C:T:61...
1,alt:rs983392:PEAKCENTER:chr11:59923508:A:G:599...,ref:rs983392:PEAKCENTER:chr11:59923508:A:G:599...
2,alt:rs965034941:SNPCENTER:chr19:1999195:CCA:C:...,ref:rs965034941:SNPCENTER:chr19:1999195:CCA:C:...
3,alt:rs953471:PEAKCENTER:chr9:124221903:G:A:124...,ref:rs545732955:PEAKCENTER:chr9:124221781:A:C:...
4,alt:rs9478143:PEAKCENTER:chr6:150862035:A:G:15...,ref:rs113543131:PEAKCENTER:chr6:150862117:C:T:...
...,...,...
850,alt:cg05066959:SNPCENTER:chr8:41519308:C:G:415...,ref:cg05066959:SNPCENTER:chr8:41519308:C:G:415...
851,alt:cg05030077:SNPCENTER:chr16:2255199:C:G:225...,ref:cg05030077:SNPCENTER:chr16:2255199:C:G:225...
852,alt:cg03169557:SNPCENTER:chr16:89598950:C:G:89...,ref:cg03169557:SNPCENTER:chr16:89598950:C:G:89...
853,alt:cg03073402:SNPCENTER:chr19:42927676:C:G:42...,ref:cg03073402:SNPCENTER:chr19:42927676:C:G:42...


In [32]:
df_index = pd.read_csv('/media/zihengc/T7/mpra3_lib_analysis/indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv' ,index_col=0)

In [33]:
df_500 = pd.DataFrame()

In [34]:
df_500['alt']=fasta_df.loc[df_alref['0']]['Sequence']
df_500['ref']=fasta_df.loc[df_alref['1']]['Sequence'].tolist()

In [35]:
df_500 = df_500.sort_index()
df_500['REFALT_Flip'] = df_index[['REFALT_Flip']]

In [36]:
df_500

,alt,ref,REFALT_Flip
enhancer_id,,,
alt:cg03073402:SNPCENTER:chr19:42927676:C:G:42927563:42927789:42927676,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:cg03169557:SNPCENTER:chr16:89598950:C:G:89598837:89599063:89598950,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:cg05030077:SNPCENTER:chr16:2255199:C:G:2255086:2255312:2255199,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:cg05066959:SNPCENTER:chr8:41519308:C:G:41519195:41519421:41519308,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:cg05228284:SNPCENTER:chr19:2720847:C:G:2720734:2720960:2720847,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
...,...,...,...
alt:rs9478143:PEAKCENTER:chr6:150862035:A:G:150861632:150862271:150862051,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:rs953471:PEAKCENTER:chr9:124221903:G:A:124221758:124221963:124221854,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False
alt:rs965034941:SNPCENTER:chr19:1999195:CCA:C:1999082:1999308:1999195,NCGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAA...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,False


In [37]:
df_500.loc[df_500['REFALT_Flip'] == True, ['alt', 'ref']] = df_500.loc[df_500['REFALT_Flip'] == True, ['ref', 'alt']].values

In [38]:
df_500 = df_500.drop("REFALT_Flip",axis=1)

In [39]:
df_500.columns = ['Minor',"Major"]

In [40]:
df_500.to_csv("indexing/500bp_major_minor.csv")

In [41]:
df_500

,Minor,Major
enhancer_id,,
alt:cg03073402:SNPCENTER:chr19:42927676:C:G:42927563:42927789:42927676,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:cg03169557:SNPCENTER:chr16:89598950:C:G:89598837:89599063:89598950,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:cg05030077:SNPCENTER:chr16:2255199:C:G:2255086:2255312:2255199,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:cg05066959:SNPCENTER:chr8:41519308:C:G:41519195:41519421:41519308,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:cg05228284:SNPCENTER:chr19:2720847:C:G:2720734:2720960:2720847,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
...,...,...
alt:rs9478143:PEAKCENTER:chr6:150862035:A:G:150861632:150862271:150862051,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs953471:PEAKCENTER:chr9:124221903:G:A:124221758:124221963:124221854,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...
alt:rs965034941:SNPCENTER:chr19:1999195:CCA:C:1999082:1999308:1999195,NCGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAA...,CGGCATGGACGAGCTGTACAAGGGAAGCCCCAAGAAAAAGCGGAAG...


In [42]:
snp_data_multi_modified["Sequence_Major_500bp"] = df_500['Major']
snp_data_multi_modified["Sequence_Minor_500bp"] = df_500['Minor']

In [43]:
snp_mac_reduce = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 2].index]
snp_mac_increase = snp_data_multi_modified.loc[df_cluster[df_cluster['Cluster'] == 3].index]

In [44]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row["Sequence_Major_500bp"]):
        record = SeqRecord(
            Seq(row["Sequence_Major_500bp"]),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row["Sequence_Minor_500bp"]):
        record = SeqRecord(
            Seq(row["Sequence_Minor_500bp"]),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('mpra3_meme_scanning/snp_reduce_atac_ml_clusterblue_major_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [45]:
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
from Bio import SeqIO
import pandas as pd
# Create SeqRecord objects
seq_records = []
data_to_save = snp_mac_reduce
for index, row in data_to_save.iterrows():
    if pd.notnull(row["Sequence_Minor_500bp"]):
        record = SeqRecord(
            Seq(row["Sequence_Minor_500bp"]),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)

data_to_save = snp_mac_increase
for index, row in data_to_save.iterrows():
    if pd.notnull(row["Sequence_Major_500bp"]):
        record = SeqRecord(
            Seq(row["Sequence_Major_500bp"]),
            id=row['SNP_ID'],
            description=f"{row['chromosome']}:{row['position']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('mpra3_meme_scanning/snp_reduce_atac_ml_clusterblue_minor_25bp_500bppadded.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Clustered SNPs from 6.1

In [11]:
#THP1 Macrophage
import pandas as pd
snp_data = pd.read_csv('indexing/snp_data_modified.csv',index_col=0)
df_allele = pd.read_csv('allele_clustering/normalized_snp_mpra_ml_cluster_fixed_20241024.csv',index_col=0)
df_allele['snp_25bp_sequence'] = snp_data['Sequence'].tolist()
df_haploreg = pd.read_csv("indexing/SNP_haploregannotatr_ATAC_tss_SNPFunction_spliceai_bayesianML_Bellenguez_202401021.csv",index_col=0)
df_allele =  pd.merge(df_allele[['Cluster','snp_25bp_sequence']],df_haploreg[['chr','RSID','pos_hg38']],left_index=True,right_index=True)

In [12]:
snp_data

,SNP_ID,chromosome,position,Major,Minor,a1,a2,Sequence,hg38_center_base,Major_len,Minor_len,Sequence_Major,Sequence_Minor,Padded_Sequence,Is_Valid_Length,Sequence_Major_padded,Sequence_Minor_padded
0,cg03073402,chr19,42423524,C,G,C,G,GCTCCGTTCCCCCGGCCAACTCCAT,C,1,1,GCTCCGTTCCCCCGGCCAACTCCAT,GCTCCGTTCCCCGGGCCAACTCCAT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
1,cg03169557,chr16,89532542,C,G,C,G,CATCGATGAGATCGACGCGGTGGGC,C,1,1,CATCGATGAGATCGACGCGGTGGGC,CATCGATGAGATGGACGCGGTGGGC,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
2,cg05030077,chr16,2205198,C,G,C,G,CAGGGAGGGCACCGGAACAGCGCGA,C,1,1,CAGGGAGGGCACCGGAACAGCGCGA,CAGGGAGGGCACGGGAACAGCGCGA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
3,cg05066959,chr8,41661790,C,G,C,G,CAGGGCCGCGGGCGCCTCAGTACCT,C,1,1,CAGGGCCGCGGGCGCCTCAGTACCT,CAGGGCCGCGGGGGCCTCAGTACCT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
4,cg05228284,chr19,2720849,C,G,C,G,AGAGGCACAGTGCGGGGCCAGCCCA,C,1,1,AGAGGCACAGTGCGGGGCCAGCCCA,AGAGGCACAGTGGGGGGCCAGCCCA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,rs9478143,chr6,150862035,A,G,A,G,GTGGCTGGAGGCAAGATATGCTGGC,A,1,1,GTGGCTGGAGGCAAGATATGCTGGC,GTGGCTGGAGGCGAGATATGCTGGC,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
851,rs953471,chr9,124221903,G,A,G,A,CAAAGGGTGGGTGGAGCTGTCGAGA,G,1,1,CAAAGGGTGGGTGGAGCTGTCGAGA,CAAAGGGTGGGTAGAGCTGTCGAGA,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
852,rs965034941,chr19,1999195,CCA,C,CCA,C,CTTGACTGCACCCCAGAGTCAGGTC,C,3,1,CTTGACTGCACCCCAGAGTCAGGTC,CTTGACTGCACCCGAGTCAGGTC,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,True,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...
853,rs983392,chr11,60156035,A,G,A,G,CTAAAATTCCACATGCAATTTTATT,A,1,1,CTAAAATTCCACATGCAATTTTATT,CTAAAATTCCACGTGCAATTTTATT,NaN,NaN,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...,NNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNNN...


In [8]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==1].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster1_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=1].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster1_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [14]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==2].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster2_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=2].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster2_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

In [15]:
# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']==3].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Cluster3_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')

# Create SeqRecord objects
seq_records = []
data_to_save = df_allele[df_allele['Cluster']!=3].drop_duplicates('RSID')
for index, row in data_to_save.iterrows():
    if pd.notnull(row['snp_25bp_sequence']):
        record = SeqRecord(
            Seq(row['snp_25bp_sequence']),
            id=row['RSID'],
            description=f"{row['chr']}:{row['pos_hg38']}"
        )
        seq_records.append(record)
# Write to a FASTA file
with open('Excluding_Cluster3_snps_25bpsequence.fasta', 'w') as fasta_file:
    SeqIO.write(seq_records, fasta_file, 'fasta')